# Class 07 · Part 1: Machine learning, a first model

**Machine learning** (ML) lets the computer learn a rule from examples, instead of us writing the rule.
In this class it is one more tool for your climate data. In this notebook you

1. learn the words of ML: features, target, training and testing,
2. build a first model: predict **tomorrow's temperature** in Hamburg from the last days,
3. **evaluate** it, and compare it with simple guesses,
4. see what **overfitting** is, and a mistake to avoid.

In part 2, `class07_ml_bias_correction.ipynb`, you use ML to correct the bias of ERA5.

## 1. Machine learning in a nutshell

**Supervised learning**: you give the computer examples where you know the answer, it learns a rule, and
then uses the rule where you do not know the answer.

| Word | Meaning | Here |
|---|---|---|
| **features** (X) | what the model sees | the temperature 1 and 2 days ago |
| **target** (y) | what it should predict | the temperature today |
| **regression** | the target is a number | a temperature |
| **classification** | the target is a class | e.g. "heavy rain" or "no heavy rain" |
| **training** | the examples the model learns from | 1936–2009 |
| **testing** | examples kept apart, to check the model | 2010–2025 |

(**Unsupervised** learning finds structure without known answers, e.g. groups of similar weather days; not in
this class.)

A model is only good if it works on data it has **not seen**: that is **generalization**. A model that learns
its training examples by heart, but fails on new data, is **overfitting**.

The package `scikit-learn` (`sklearn`) has many models, all used in the same two steps:

```python
model.fit(X_train, y_train)          # learn from the training examples
prediction = model.predict(X_test)   # use the rule on new data
```

## 2. Features and target

The daily mean temperature of Hamburg-Fuhlsbüttel, from class 02 (the file is in `data/` already, or it is
downloaded). `.shift(1)` moves a column one day down: on each day you see the value of the day before.

In [ ]:
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils import download_dwd_daily


def read_station(number):
    # the daily data of a DWD station, with dates as index (class 02)
    zip_file = zipfile.ZipFile(download_dwd_daily(number))
    data_name = [
        name for name in zip_file.namelist() if name.startswith("produkt_klima_tag")
    ][0]
    data = pd.read_csv(
        zip_file.open(data_name), sep=";", skipinitialspace=True, na_values=[-999]
    )
    data["date"] = pd.to_datetime(data["MESS_DATUM"].astype(str), format="%Y%m%d")
    return data.set_index("date")


hamburg = read_station(1975)

data = pd.DataFrame({"today": hamburg["TMK"]})
data["yesterday"] = data["today"].shift(1)
data["two_days_ago"] = data["today"].shift(2)
data = data.dropna()  # the first two days have no "yesterday"
data.head()

Each row is one **sample**: two features and the target. `X` is the table of features, `y` the target:

In [ ]:
X = data[["yesterday", "two_days_ago"]]
y = data["today"]
print(len(data), "samples")

## 3. Training and testing, in time order

We train on 1936–2009 and test on 2010–2025. For time series the split must follow **time**, not be random:

- A forecast only knows the **past**: the test should be in the future of the training, as in real use.
- The **climate changes**: the test years are warmer. A test in later years checks whether the model still
  works in a warmer climate; a random mix of all years could not tell.

In [ ]:
train = data.index < "2010-01-01"
test = ~train  # ~ means "not"

print(f"training: {train.sum()} days, mean {y[train].mean():.2f} °C")
print(f"testing:  {test.sum()} days, mean {y[test].mean():.2f} °C")

## 4. Train a model

A **linear regression**: today = $a$ × yesterday + $b$ × two days ago + $c$. `fit` finds the best $a$, $b$, $c$
on the training days.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X[train], y[train])

print("a (yesterday):    ", round(model.coef_[0], 3))
print("b (two days ago): ", round(model.coef_[1], 3))
print("c:                ", round(model.intercept_, 3))

Almost all the weight is on yesterday: tomorrow is mostly like today.

## 5. Evaluate

The **RMSE** (root mean squared error) is the typical size of the error, in °C. A number alone says little: compare
it with simple guesses that need no ML:

- **persistence**: tomorrow = today,
- **climatology**: always the mean temperature of the training days.

In [ ]:
from sklearn.metrics import mean_squared_error


def rmse(observed, predicted):
    return np.sqrt(mean_squared_error(observed, predicted))


prediction = model.predict(X[test])

print(f"linear regression: {rmse(y[test], prediction):.2f} °C")
print(f"persistence:       {rmse(y[test], X['yesterday'][test]):.2f} °C")
print(
    f"climatology:       {rmse(y[test], np.full(test.sum(), y[train].mean())):.2f} °C"
)

The model is much better than climatology, but only a little better than persistence: with two days of the
same station there is not much more to learn. That is a normal, honest result: always compare with a simple guess.

Observations and predictions of one test year:

In [ ]:
predicted = pd.Series(prediction, index=y[test].index)

fig, axes = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={"width_ratios": [2, 1]})
y[test].loc["2024"].plot(ax=axes[0], label="observed", color="black")
predicted.loc["2024"].plot(ax=axes[0], label="predicted", color="firebrick")
axes[0].set_ylabel("temperature / °C")
axes[0].set_title("Hamburg, daily mean temperature, 2024")
axes[0].legend()

axes[1].scatter(y[test], predicted, s=2, alpha=0.3)
axes[1].plot(
    [-15, 30], [-15, 30], color="black"
)  # perfect predictions lie on this line
axes[1].set_xlabel("observed / °C")
axes[1].set_ylabel("predicted / °C")
axes[1].set_title("All test days, 2010–2025")

fig.savefig(
    "/app/pages/07_machine_learning/regression.png", dpi=100, bbox_inches="tight"
)
plt.show()

## 6. Overfitting

A more flexible model: a **decision tree** asks yes/no questions about the features ("was yesterday warmer
than 12.3 °C?") and predicts the mean of the training days that end in the same leaf. `max_depth` limits the
number of questions. Compare its error on the **training** days and on the **test** days:

In [ ]:
from sklearn.tree import DecisionTreeRegressor

depths = [1, 2, 3, 5, 8, 12, 16, 20, None]  # None: no limit
train_error, test_error = [], []
for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(
        X[train], y[train]
    )
    train_error.append(rmse(y[train], tree.predict(X[train])))
    test_error.append(rmse(y[test], tree.predict(X[test])))

labels = [str(d) for d in depths]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(labels, train_error, marker="o", label="training days")
ax.plot(labels, test_error, marker="o", label="test days")
ax.axhline(
    rmse(y[test], prediction),
    color="black",
    linestyle="--",
    label="linear regression, test",
)
ax.set_xlabel("max_depth of the tree")
ax.set_ylabel("RMSE / °C")
ax.set_title("Overfitting: the deeper the tree, the better it knows its training days")
ax.legend()

fig.savefig(
    "/app/pages/07_machine_learning/overfitting.png", dpi=100, bbox_inches="tight"
)
plt.show()

A deep tree learns the training days by heart: its training error falls, its test error grows. A model that
looks great on its own training data can be bad on new data: always evaluate on data the model has not seen.

## 7. A mistake to avoid: information from the future

`.shift(-1)` moves a column **up**: it gives each day the temperature of **tomorrow**. Used as a feature by
mistake, it makes the model look much better:

In [ ]:
wrong = data.copy()
wrong["tomorrow"] = wrong["today"].shift(-1)  # the future: not known when you forecast!
wrong = wrong.dropna()
wrong_train = wrong.index < "2010-01-01"
features = ["yesterday", "two_days_ago", "tomorrow"]

leaky = LinearRegression().fit(
    wrong[features][wrong_train], wrong["today"][wrong_train]
)
print(
    f"with tomorrow as a feature: {rmse(wrong['today'][~wrong_train], leaky.predict(wrong[features][~wrong_train])):.2f} °C"
)

Better, and useless: on the day you forecast, you do not know tomorrow. If a result looks too good, check
that no feature contains information from the future (this is called **leakage**).

## 8. Exercises

1. **More features:** add the temperature three days ago. Does the model beat persistence by more?
2. **Further ahead:** predict the temperature in **two days** (features: today and yesterday; target:
   `data["today"].shift(-2)` is wrong as a feature but right as a target: why?). How does the RMSE grow?
3. **Another station:** the Zugspitze, number 5792, on top of a mountain. Is it easier or harder to predict?

In [ ]:
# 1. more features

In [ ]:
# 2. two days ahead

In [ ]:
# 3. another station

## Next

Open **`class07_ml_bias_correction.ipynb`**: machine learning to correct the bias of a reanalysis.